# Nulling a source directly from arrays

`jolly_roger`'s tapering core operates on plain arrays, so you can drive it
without a measurement set. This page builds a synthetic dataset with a source
whose delay drifts with time, nulls it with
{func}`~jolly_roger.tractor.compute_tukey_multi_taper`, then walks through the
options that shape what gets nulled. The page is executed when the docs are
built, so every plot below is the real output.

## Real geometry, still without a measurement set

Here `WDelays` was built by hand from a made-up track. For a physical source,
the same object comes from the array-based geometry core
{func}`~jolly_roger.uvws.get_object_delay`, fed by `get_baselines(ant_xyz)` and
`make_hour_angles(times, location, position)` -- none of which touch casacore.
Pass a `SkyCoord` (or `"sun"`) as the position to stay fully offline.

In [ ]:
import logging

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from numpy import ma

from jolly_roger.delays import data_to_delay_time
from jolly_roger.logging import logger
from jolly_roger.tractor import (
    DataChunk,
    TukeyTractorOptions,
    compute_tukey_multi_taper,
)
from jolly_roger.uvws import WDelays

logger.setLevel(logging.WARNING)  # keep the taper's info logging out of the output

## A synthetic drifting source

We work in delay space: a faint noise background, a steady source at delay 0
(the field, which should survive), and one source whose delay follows a sine
across time. The drifting source is a couple of delay bins wide and brightens
over the observation, so the brightness-dependent options below have something
to bite on. Two helpers build the dataset and the delay description; the taper
only cares about a handful of `DataChunk` fields, so the rest default away.

In [ ]:
rng = np.random.default_rng(1234)
n_time, n_chan, n_pol = 256, 512, 4
freq_chan = np.linspace(744, 1032, n_chan) * u.MHz

# The delay axis matching jolly_roger's forward transform.
delay = np.fft.fftshift(np.fft.fftfreq(n_chan, d=np.diff(freq_chan).mean()).decompose())
delay_ns = delay.to(u.ns).value


def make_dataset(offset_ns=0.0):
    """A drifting source (optionally offset from its predicted delay) plus a
    steady field at delay 0, as a frequency-time DataChunk."""
    background = rng.normal(loc=0, scale=1, size=(n_time, n_chan)) + 1j * rng.normal(
        size=(n_time, n_chan)
    )
    delay_space = np.fft.fftshift(
        np.fft.fft(background, axis=1, norm="forward"), axes=1
    )

    track_ns = 0.3 * delay_ns.max() * np.sin(np.linspace(0, 2 * np.pi, n_time))
    track_bin = np.abs(delay_ns[None, :] - (track_ns + offset_ns)[:, None]).argmin(
        axis=1
    )
    brightness = np.linspace(0.0, 1.0, n_time)
    half_width = 1
    rows = np.arange(n_time)
    for offset in range(-half_width, half_width + 1):
        delay_space[rows, np.clip(track_bin + offset, 0, n_chan - 1)] += brightness

    zero_bin = np.abs(delay_ns).argmin()
    for offset in range(-half_width, half_width + 1):
        delay_space[rows, np.clip(zero_bin + offset, 0, n_chan - 1)] += 1.0

    vis = np.fft.ifft(np.fft.ifftshift(delay_space, axes=1), axis=1, norm="forward")
    chunk = DataChunk(
        masked_data=ma.masked_array(
            vis[..., None].repeat(n_pol, -1),
            mask=np.zeros((n_time, n_chan, n_pol), bool),
        ),
        freq_chan=freq_chan,
        time_mjds=np.arange(n_time, dtype=float),
        ant_1=np.zeros(n_time, dtype=np.int64),
        ant_2=np.ones(n_time, dtype=np.int64),
        chunk_size=n_time,
    )
    return chunk, track_ns, track_bin, brightness


def make_wdelays(track_ns, guard_ns=None):
    """The per-time delay of the source to null, plus the trivial single-baseline
    index maps. ``guard_ns`` sets a protected half-width around delay 0."""
    guard = None if guard_ns is None else np.full((1, n_time), guard_ns * 1e-9) * u.s
    return WDelays(
        object_name="drifting-source",
        w_delays=(track_ns * u.ns).to(u.s)[None, :],  # shape (baseline=1, time)
        b_map={(0, 1): 0},
        time_map={t * u.s: idx for idx, t in enumerate(np.arange(n_time, dtype=float))},
        elevation=np.full(n_time, 90.0) * u.deg,
        guard_region=guard,
    )


def flagged_amp(delay_time, result):
    """|delay spectrum| (pol 0) with flagged time steps blanked to NaN."""
    amp = np.abs(delay_time.delay_time[..., 0])
    if result.flags is not None:
        amp = np.where(result.flags[:, 0, 0][:, None], np.nan, amp)
    return amp

## Apply the taper

In [ ]:
chunk, track_ns, track_bin, brightness = make_dataset()
vis_before = np.asarray(chunk.masked_data[..., 0])
before = data_to_delay_time(chunk)

options = TukeyTractorOptions(outer_width_ns=40.0, tukey_width_ns=30.0)
result = compute_tukey_multi_taper(chunk, options, [make_wdelays(track_ns)])

after = data_to_delay_time(result.data_chunk)
vis_after = np.asarray(result.data_chunk.masked_data[..., 0])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
panels = [
    (np.abs(before.delay_time[..., 0]), "Before"),
    (flagged_amp(after, result), "After"),
]
for ax, (amp, title) in zip(axes, panels, strict=True):
    im = ax.pcolormesh(
        np.arange(n_time), delay_ns, amp.T, shading="auto", vmin=0, vmax=1
    )
    ax.set(title=title, xlabel="time index", ylabel="delay [ns]", ylim=(-350, 350))
    fig.colorbar(im, ax=ax)
fig.tight_layout()

The drifting track is gone while the steady source at delay 0 (the field)
survives. Where the drifting source sweeps through zero delay the two overlap,
so those time steps are flagged -- shown blanked here.

In [ ]:
window = np.clip(track_bin[:, None] + np.arange(-1, 2)[None, :], 0, n_chan - 1)
rows2 = np.arange(n_time)[:, None]
before_src = np.abs(before.delay_time[rows2, window, 0]).mean()
after_src = np.abs(after.delay_time[rows2, window, 0]).mean()
print(f"mean source amplitude: {before_src:.3f} -> {after_src:.4f}")
assert after_src < 0.1 * before_src

## In frequency space

The taper is applied in delay space but written back as frequency-time
visibilities. Removing the drifting source flattens the fringe it imprinted
across frequency, in both amplitude and phase. Where that source crosses the
field at delay 0 the two cannot be separated, so those times are flagged; here
we apply the flags by blanking (NaN) that data, which shows up as gaps.

In [ ]:
after_flagged = np.where(result.flags[..., 0], np.nan, vis_after)

fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True, sharey=True)
panels = [
    (np.abs(vis_before), "amplitude before"),
    (np.abs(after_flagged), "amplitude after"),
    (np.angle(vis_before, deg=True), "phase before"),
    (np.angle(after_flagged, deg=True), "phase after"),
]
for ax, (data, title) in zip(axes.ravel(), panels, strict=True):
    if "amplitude" in title:
        cmap = "viridis"
        vmin, vmax = 0, 5
    else:
        cmap = "twilight_shifted"
        vmin, vmax = -180, 180
    im = ax.pcolormesh(
        np.arange(n_time),
        freq_chan.value,
        data.T,
        shading="auto",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )
    ax.set(title=title)
    fig.colorbar(im, ax=ax)
for ax in axes[:, 0]:
    ax.set_ylabel("frequency [MHz]")
for ax in axes[1, :]:
    ax.set_xlabel("time index")
fig.tight_layout()

## `compare_to_field`: only null sources brighter than the field

With `compare_to_field` set, a time step is only nulled if the source is
brighter (in delay space) than that fraction of the field. Here the source
brightens over time, so its faint early half is left alone and only the bright
later half is removed.

In [ ]:
chunk, track_ns, track_bin, brightness = make_dataset()
before_c = data_to_delay_time(chunk)
result_c = compute_tukey_multi_taper(
    chunk,
    TukeyTractorOptions(outer_width_ns=40.0, tukey_width_ns=30.0, compare_to_field=0.2),
    [make_wdelays(track_ns)],
)
after_c = data_to_delay_time(result_c.data_chunk)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
panels = [
    (np.abs(before_c.delay_time[..., 0]), "Before"),
    (flagged_amp(after_c, result_c), "After"),
]
for ax, (amp, title) in zip(axes, panels, strict=True):
    im = ax.pcolormesh(
        np.arange(n_time), delay_ns, amp.T, shading="auto", vmin=0, vmax=1
    )
    ax.set(title=title, xlabel="time index", ylabel="delay [ns]", ylim=(-350, 350))
    fig.colorbar(im, ax=ax)
fig.tight_layout()

## `peak_shift_search`: recover a mispredicted delay

If the source sits away from its predicted delay (ionospheric shift, an
imperfect model), a fixed taper misses it. `peak_shift_search` looks for the
real peak near the prediction and slides the null onto it. Here the source is
offset 60 ns from the prediction -- more than the taper is wide -- so the plain
taper leaves it behind while the peak search removes it.

In [ ]:
chunk, track_ns, track_bin, _ = make_dataset(offset_ns=60.0)
before_p = data_to_delay_time(chunk)
result_plain = compute_tukey_multi_taper(chunk, options, [make_wdelays(track_ns)])
after_plain = data_to_delay_time(result_plain.data_chunk)

chunk, track_ns, track_bin, _ = make_dataset(offset_ns=60.0)
result_peak = compute_tukey_multi_taper(
    chunk,
    TukeyTractorOptions(
        outer_width_ns=40.0,
        tukey_width_ns=30.0,
        peak_shift_search=True,
        peak_shift_search_width_ns=120.0,
    ),
    [make_wdelays(track_ns)],
)
after_peak = data_to_delay_time(result_peak.data_chunk)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)
panels = (
    (np.abs(before_p.delay_time[..., 0]), "Before"),
    (flagged_amp(after_plain, result_plain), "Plain taper"),
    (flagged_amp(after_peak, result_peak), "peak_shift_search"),
)
for ax, (amp, title) in zip(axes, panels, strict=True):
    im = ax.pcolormesh(
        np.arange(n_time), delay_ns, amp.T, shading="auto", vmin=0, vmax=1
    )
    ax.set(title=title, xlabel="time index", ylim=(-400, 400))
    fig.colorbar(im, ax=ax)
axes[0].set_ylabel("delay [ns]")
fig.tight_layout()